# Chapter 20 — Fusion

Part 7: Kernel construction and optimization.

**Thread.**
- **Recurring idea.** Fusion moves intermediates from HBM to registers; it changes traffic, not the calculus.
- **Topic chain.** Separate kernels → fused elementwise maps → fused reduction/normalization → fused GEMM epilogue → the register-pressure limit.
- **Tension.** Every extra live tile saves a round trip but consumes registers; the byte model gives the ceiling, and occupancy and spills decide the result.
- **Check.** Count full-size streams before writing a kernel.

**Ladder.**
1. **Method 01 — three separate kernels.** Eager PyTorch runs $t=xa$, $u=t+b$, and $y=\operatorname{ReLU}(u)$ as three launches with two full-size intermediates in HBM.
2. **Method 02 — one fused elementwise kernel.** A masked load, the whole composition in registers, and one masked store.
3. **Method 03 — a fused GEMM epilogue.** `tl.dot` accumulates in fp32 registers; bias and activation run before the only store.

**Study tip.**
- For $N=1024$ `float32` ($b=4$ bytes), count full-size streams.
- Three passes move $2Nb+2Nb+2Nb=6Nb=24{,}576$ bytes.
- One fused pass moves $Nb+Nb=2Nb=8{,}192$ bytes.
- The fusion saves $4Nb=16{,}384$ bytes, a factor of three, with identical arithmetic.
- **By hand.** Write that table before reading any kernel.

**Learning objectives**
- State the traffic rule $Q_{\text{fused}}=(I+O)Nb$ and compare it with $Q_{\text{sep}}=2kNb$ for a chain of $k$ passes.
- Read a composition as register dataflow: load once, compose, store once.
- Fuse bias, activation, residual, and multi-op pointwise chains.
- Fuse a reduction with its normalization and a bias-plus-activation epilogue into a GEMM accumulator.
- Decide when *not* to fuse from register pressure, occupancy, recomputation, code size, and numerical differences.

[Index](../../00_index.ipynb) · [← Previous](../ch19_tiling/19_complete.ipynb) · [Next →](../../part8_advanced/ch21_atomics/21_complete.ipynb)

**Estimated time:** 75–90 min · **Difficulty:** Advanced

**Prerequisites**
- Chapter 19: tiling, block size, warps per program, and register blocking.
- Chapter 13: coalescing, bandwidth, and register/shared-memory trade-offs.
- Chapter 7: elementwise arithmetic; Chapter 8: masks and `tl.where`.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

### 1. Separate kernels: every intermediate goes through HBM

```text
HBM x  ──load──▶ kernel 1 ──store──▶ HBM t1
HBM t1 ──load──▶ kernel 2 ──store──▶ HBM t2
HBM t2 ──load──▶ kernel 3 ──store──▶ HBM y
```

- **Traffic.** Three passes move $2Nb+2Nb+2Nb=6Nb$.
- **Arithmetic.** Identical to the fused version; only the trips change.

### 2. Fused dataflow: the intermediates stay in registers

```text
HBM x ──load──▶ registers: t1 ▶ t2 ▶ ... ▶ y ──store──▶ HBM y
```

- **Traffic.** Read each input once, write each output once: $Q=(I+O)Nb$.
- **Unary chain.** $I=O=1$, so $Q=2Nb$.

### 3. A GEMM epilogue runs on the accumulator

```text
A[M,K] ─┐
        ├─ tl.dot ─▶ acc[M,N] in registers ─+bias─▶ act ─▶ C[M,N]
B[K,N] ─┘
```

- **Key point.** The $M\times N$ matrix never exists in HBM before the epilogue.
- **Output.** Only $C$ is written.

### 4. Register pressure: the price of a longer fused chain

```text
live tiles L:   1      2      4      8     16     32
regs/thread:    8     16     32     64    128    256
                                              └── spill above 255
```

- **Assumptions.** $E=1024$ fp32 elements per program, $T=128$ threads.
- **Spill cost.** A spilled tile is written to and read back from local memory, adding $2Nb$ per tile.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

torch.manual_seed(0)
print("device:", DEVICE, "| GPU:", HAS_GPU)

# Part I — Mathematics and concepts

- **Notation.** $N$ elements, $b$ bytes per element, $I$ full-size input streams, $O$ full-size output streams.
- **Traffic rule.** A kernel that touches only those streams moves $Q=(I+O)Nb$.
- **Separate chain.** $k$ pointwise passes move $Q_{\text{sep}}=2kNb$: each pass reads one full vector and writes one full vector.
- **Counting convention.** Count full-size streams only, where the kernel actually reads them; a scalar is a register, and a length-$N$ broadcast row is small next to an $M\times N$ intermediate.
- **Reading order.** Each subsection follows Connection → Rule → Formula → Example → Where → Triton → Failure → Bridge.

### 20.1 Add + multiply

**Connection.** Builds on Chapter 7's elementwise lift and Chapter 19's one-load/one-store tile.

**Rule.** Two pointwise maps can share one kernel when the intermediate is not reused elsewhere.

**Formula.** With scalars $b$ and $s$, $Y=(X+b)s$.

**Example.** For $x=(1,2,3,4)$, $b=1$, $s=2$: $t=x+b=(2,3,4,5)$ and $y=ts=(4,6,8,10)$.

**Bytes.** Unfused: $2Nb$ for the add plus $2Nb$ for the multiply, or $4Nb$. Fused: one load and one store, $2Nb$. Saved: $2Nb$.

**Where.** Scale-and-shift inside normalization layers, the linear part of a residual branch, and a temperature or learning-rate schedule applied to a whole tensor.

**Triton.** Keep both scalars as runtime arguments and write `y = (x + b) * s` between one masked load and one masked store; both live in registers, not HBM.

**Failure.** Storing $t$ "for later" restores $4Nb$. If $b$ is a full vector, the unfused count is $5Nb$ and the fused count is $3Nb$, not $2Nb$.

**Bridge.** Applying the affine in the other order gives §20.2; the byte count is unchanged.

### 20.2 Multiply + add

**Connection.** Reorders the maps from §20.1; it is the same composition with the multiply first.

**Rule.** The affine map is one FMA, with one rounding step, when the hardware supports it.

**Formula.** $Y=Xa+b$.

**Example.** For $x=(1,2,3,4)$, $a=2$, $b=-1$: $y=2x-1=(1,3,5,7)$.

**Bytes.** Unfused: $2Nb$ for the multiply plus $2Nb$ for the add, or $4Nb$. Fused: $2Nb$. Saved: $2Nb$.

**Where.** The per-channel affine of a normalization layer, the scale and shift of a residual branch, and the epilogue of nearly every linear layer.

**Triton.** Write `y = a * x + b` as one expression; the compiler forms the FMA; do not create a separate `tmp` array.

**Failure.** Applying the affine after the activation when the model specifies it before, or the reverse, changes the function even though the bytes are unchanged.

**Bridge.** Making $b$ a vector turns the scalar shift into a broadcast and leads to §20.3.

### 20.3 Bias + activation

**Connection.** Extends §20.2 by replacing the scalar $b$ with a length-$N$ bias vector.

**Rule.** A bias vector broadcasts over the element axis, then the activation acts lane by lane.

**Formula.** $Y=\operatorname{act}(X+B)$ with $B\in\mathbb R^{N}$.

**Example.** For $x=(-2,0.5,2)$, $b=(1,1,1)$, and $\operatorname{act}=\operatorname{ReLU}$: $z=x+b=(-1,1.5,3)$ and $y=\operatorname{ReLU}(z)=(0,1.5,3)$.

**Bytes.** Unfused: $3Nb$ for the bias add plus $2Nb$ for the activation, or $5Nb$. Fused: read $x$ and $b$, write $y$, or $3Nb$. Saved: $2Nb$.

**Where.** A linear layer followed by ReLU, GELU, SiLU, or tanh; the bias is the vector a GEMM epilogue adds before the activation.

**Triton.** In 2-D, load the bias with the column offsets and add it as `b[None, :]`, which broadcasts over rows without materializing a matrix.

**Failure.** `b[:, None]` broadcasts over columns, so a length-$N$ bias silently becomes a per-row bias; applying the activation before the bias is a different function.

**Bridge.** Replacing the broadcast bias by a same-shape tensor gives the residual form in §20.4.

### 20.4 Residual + activation

**Connection.** Generalizes §20.3 from a broadcast bias to a full-size residual stream.

**Rule.** The residual has the same shape as $X$, so the add is lane by lane.

**Formula.** $Y=\operatorname{act}(X+R)$.

**Example.** For $x=(-1,0.5)$ and $r=(0.5,-1)$: $z=x+r=(-0.5,-0.5)$ and $y=\operatorname{ReLU}(z)=(0,0)$.

**Bytes.** Unfused: $3Nb$ for the residual add plus $2Nb$ for the activation, or $5Nb$. Fused: read $x$ and $r$, write $y$, or $3Nb$. Saved: $2Nb$.

**Where.** A residual block, a skip connection around an MLP, and the "add the branch back" step of a transformer layer.

**Triton.** Pass a second pointer with the same offsets and mask; the two loads are independent, so the compiler can overlap them with the arithmetic.

**Failure.** Writing in place over $X$ when $R$ aliases $X$, or discarding $X+R$ when the backward pass still needs the pre-activation value.

**Bridge.** Repeating the same load-compose-store pattern for several maps gives the chain in §20.5.

### 20.5 Multiple elementwise operations

**Connection.** Iterates the single-map pattern from §20.4 into a chain of maps.

**Rule.** A pointwise chain is one fused kernel: load $x$, apply every $f_i$ in registers, store $y$.

**Formula.** $Y=f_k(\cdots f_2(f_1(X)))$.

**Example.** For $x=(0,1)$ and $f(x)=1/(1+\sqrt{\lvert 2(x+1)\rvert+1})$:
- $x+1=(1,2)$ and $2(x+1)=(2,4)$.
- $\lvert\cdot\rvert+1=(3,5)$ and $\sqrt{\cdot}=(\sqrt3,\sqrt5)$.
- $y=(1/(1+\sqrt3),\,1/(1+\sqrt5))\approx(0.3660254,\,0.3090170)$.

**Bytes.** Unfused: $k$ unary passes cost $2kNb$; for $k=5$, $10Nb$. Fused: $2Nb$. Saved: $2(k-1)Nb=8Nb$.

**Where.** Activation chains, normalization followed by a scale, dropout scaling followed by a mask, and pre-processing pipelines.

**Triton.** Write the chain as local expressions (`y = x + 1; y = y * 2; ...`); the compiler keeps one live tile, not $k$ copies, when it can see earlier values are dead.

**Failure.** A long chain raises register pressure and code size; reusing an early intermediate keeps it live to the end and can make one kernel slower than two shorter ones.

**Bridge.** A chain that ends in a reduction can still be one kernel; §20.6 moves the reduction inside it.

### 20.6 Reduction + normalization

**Connection.** Adds a reduction to the pointwise chain from §20.5, so the output depends on the whole tile.

**Rule.** Load the row once, reduce in registers, broadcast the scalar back over the tile, and store once.

**Formula.** For a row $X$ and $S=\sum_j X_j$, $Y=X/S$.

**Softmax.** Same pattern for $Y_j=e^{X_j-m}/\sum_i e^{X_i-m}$ with $m=\max_j X_j$: one read, one reduction, one write.

**Example.** For $x=(1,2,3,4)$: $S=1+2+3+4=10$ and $y=(0.1,\,0.2,\,0.3,\,0.4)$.

**Bytes.** Unfused: the reduction reads $X$ once ($Nb$) and writes a scalar; normalization reads $X$ again and writes $Y$ ($2Nb$); total $3Nb$. Fused: load once, reduce, write once, or $2Nb$. Saved: $Nb$.

**Where.** Softmax, LayerNorm and RMSNorm statistics, attention scores, and any "divide by the row total" normalization.

**Triton.** Load the row into a tile, call `tl.sum(x, axis=...)`, and broadcast the scalar back before the store; do not launch a second kernel just to divide.

**Failure.** Reducing the wrong axis, using `other=0.0` for a maximum when every real value is negative, dividing by the padded block width instead of the true row length, or normalizing with a stale statistic.

**Bridge.** The same register tile becomes an accumulator when the producer is a matrix product; §20.7 adds the first GEMM epilogue.

### 20.7 GEMM + bias

**Connection.** Replaces the pointwise producer of §20.6 with a matrix product and keeps the epilogue on the accumulator.

**Rule.** Add the bias to the fp32 accumulator before the only store; the intermediate $AB$ never reaches HBM.

**Formula.** For $A\in\mathbb R^{M\times K}$, $B\in\mathbb R^{K\times N}$, and $b\in\mathbb R^{N}$: $C=AB+\mathbf 1 b^\top$, with $C_{ij}=\sum_{\ell=1}^{K}A_{i\ell}B_{\ell j}+b_j$.

**Example.**

$$
A=\begin{bmatrix}1&2\\3&4\end{bmatrix},\qquad B=I_2,\qquad b=\begin{bmatrix}10&20\end{bmatrix}
\quad\Longrightarrow\quad
C=\begin{bmatrix}11&22\\13&24\end{bmatrix}.
$$

**Bytes.** Let $G=MN$. Unfused: GEMM writes $AB$ ($Gb$), then the bias kernel reads it and writes $C$ ($2Gb$); total $3Gb$. Fused: write $C$ once, $Gb$. Saved: $2Gb$.

**Where.** Every linear layer, projection, and classifier head; the bias is tiny ($Nb$), but the intermediate matrix is not.

**Triton.** Keep the accumulator in fp32, load the bias with the column offsets, add `bias[None, :]`, and only then cast and store; the epilogue is a few instructions after the final `tl.dot`.

**Failure.** Adding a column bias as `bias[:, None]`, casting the accumulator to fp16 before the bias add, or masking the bias load with the wrong bound.

**Bridge.** Applying an activation to the biased accumulator before the store gives §20.8.

### 20.8 GEMM + activation

**Connection.** Extends the epilogue of §20.7 with a lane-wise activation on the biased accumulator.

**Rule.** The activation consumes registers, not HBM, and runs before the cast to a narrow dtype.

**Formula.** $C=\operatorname{act}\left(AB+\mathbf 1 b^\top\right)$.

**Example.**

$$
A=\begin{bmatrix}1&-1\\-2&3\end{bmatrix},\qquad B=I_2,\qquad b=0
\quad\Longrightarrow\quad
\operatorname{ReLU}(AB)=\begin{bmatrix}1&0\\0&3\end{bmatrix}.
$$

**Bytes.** Unfused: GEMM writes $AB$ ($Gb$), then the activation reads and writes ($2Gb$); total $3Gb$. Fused: write $C$ once, $Gb$. Saved: $2Gb$.

**Where.** The MLP block of a transformer, a convolution followed by ReLU, and any fused linear-activation layer.

**Triton.** Apply `tl.maximum(acc, 0.0)` or the GELU formula to the fp32 accumulator, then cast.

**Failure.** Applying the activation after the cast to fp16/bf16 changes rounding; a libdevice transcendental may not be implemented by the interpreter; a masked lane must not be activated into a stored value.

**Bridge.** With both epilogues in hand, §20.9 compares their traffic with the separate-kernel baseline.

### 20.9 Fusion vs separate kernels

**Connection.** Collects the byte counts from §20.1–§20.8 into one decision rule.

**Rule.** For $k$ separate unary passes and a fused kernel with $I$ inputs and $O$ outputs, compare total bytes, not operator count.

**Formula.** $Q_{\text{sep}}=2kNb$ and $Q_{\text{fused}}=(I+O)Nb$. For a unary chain, $I=O=1$, so $Q_{\text{fused}}=2Nb$ and $Q_{\text{sep}}/Q_{\text{fused}}=k$.

**Example.** For $N=1024$, $b=4$, $k=3$: $Q_{\text{sep}}=6Nb=24{,}576$ bytes and $Q_{\text{fused}}=2Nb=8{,}192$ bytes, saving $16{,}384$ bytes, a factor of $3$.

**Where.** The decision rule for every pointwise composition in this chapter; on a memory-bound kernel, time tracks bytes.

**Triton.** Count streams before writing the kernel; a scalar or small broadcast row is not a full-size stream.

**Failure.** Promising the full ratio when the intermediate stays in cache, launch overhead dominates, or fusion lowers occupancy; measure after the model says fusion should win.

**Bridge.** The missing constraint in the byte model is register capacity; §20.10 makes it explicit.

### 20.10 Register-pressure trade-offs

**Connection.** Adds the register limit to the traffic rule of §20.9 and the tile model of Chapter 19.

**Rule.** Fuse while the live state fits the register budget; a spilled tile pays for itself twice.

**Formula.** For a tile of $E$ elements over $T$ threads and $L$ live fp32 tiles, $r\approx\lceil LE/T\rceil$ registers per thread. Spill when $r\gt R_{\max}$ (255 fp32 registers on many NVIDIA parts); each spilled tile adds $Q_{\text{spill}}=2Nb$.

**Example.** For $E=1024$, $T=128$: $L=8$ gives $r=64$; $L=32$ gives $r=256\gt 255$, so it spills. At $N=1024$, $b=4$, traffic becomes $2Nb+2Nb=4Nb=16{,}384$ bytes, a tie with a two-pass unfused chain.

**Where.** Every long fused chain, reduction, and GEMM epilogue; the trade-off is sharpest when the tile is already large.

**Triton.** Reduce `BLOCK`, increase `num_warps`, split the chain at a materialized boundary, or recompute a cheap value instead of keeping it live.

**Failure.** A spill turns a "one-pass" kernel into a four-stream kernel; lower occupancy hides less latency; recomputation and code bloat can cost more than the saved traffic.

**Bridge.** Part II measures the three-pass baseline, Part III builds the fused elementwise kernels, and Part IV applies the epilogue to `tl.dot`.

**Where this leads.**
- **Part II.** The three-kernel PyTorch baseline gives the correctness oracle and the measured byte count.
- **Part III.** Each pointwise and reduction rule becomes one masked-load/compose/masked-store kernel.
- **Part IV.** The GEMM epilogue moves bias and activation into the fp32 accumulator.
- **Chapter 21.** Atomics and synchronization take over when a fused update must be shared across programs.

## Chapter map

| § | Fusion | Formula | Bytes saved | Kernel/demo |
|---|---|---|---|---|
| **20.1** | Add + multiply | $Y=(X+b)s$ | $2Nb$ | `fused_affine_relu` |
| **20.2** | Multiply + add | $Y=Xa+b$ | $2Nb$ | `fused_affine_relu` |
| **20.3** | Bias + activation | $Y=\operatorname{act}(X+B)$ | $2Nb$ | Method 01 / Method 02 |
| **20.4** | Residual + activation | $Y=\operatorname{act}(X+R)$ | $2Nb$ | `fused_residual_relu` |
| **20.5** | Multiple elementwise ops | $Y=f_k(\cdots f_1(X))$ | $2(k-1)Nb$ | `fused_chain` |
| **20.6** | Reduction + normalization | $Y=X/\sum_j X_j$ | $Nb$ | `fused_rms_norm` |
| **20.7** | GEMM + bias | $C=AB+\mathbf 1b^\top$ | $2MNb$ | `gemm_epilogue` |
| **20.8** | GEMM + activation | $C=\operatorname{act}(AB+\mathbf 1b^\top)$ | $2MNb$ | `gemm_epilogue` |
| **20.9** | Fusion vs separate kernels | $Q_{\text{sep}}=2kNb$ | $2(k-1)Nb$ | traffic model |
| **20.10** | Register-pressure trade-offs | $r\approx LE/T$ | spill adds $2Nb$ | register model |

## Worked numerical example

- **Spine.** $x=(1,2,3,4)$, $a=2$, $b=-1$.
- **Affine.** $y=ax+b=2x-1=(1,3,5,7)$.
- **Activation.** Every entry is positive, so $\operatorname{ReLU}(y)=(1,3,5,7)$; the activation is inactive on this spine.
- **Three passes.** $t_1=ax$, $t_2=t_1+b$, and $y=\operatorname{ReLU}(t_2)$.
- **Separate traffic.** $Q_{\text{sep}}=2Nb+2Nb+2Nb=6Nb=24{,}576$ bytes for $N=1024$ fp32.
- **Fused traffic.** Load $x$ once, compute $ax+b$ and the ReLU in registers, store $y$ once: $Q_{\text{fused}}=Nb+Nb=2Nb=8{,}192$ bytes.
- **Traffic table.**

| Path | Loads | Stores | Bytes at $N=1024$, $b=4$ | Relative |
|---|---|---:|---:|---:|
| three separate kernels | $3Nb$ | $3Nb$ | $24{,}576$ | $3.00\times$ |
| one fused kernel | $Nb$ | $Nb$ | $8{,}192$ | $1.00\times$ |
| saved | $2Nb$ | $2Nb$ | $16{,}384$ | $3.00\times$ less |

- **Check.** The next cell verifies $y=(1,3,5,7)$ and every byte count with assertions only.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = 2.0 * x - 1.0
assert x.shape == (4,)
assert y.tolist() == [1.0, 3.0, 5.0, 7.0]
assert torch.equal(torch.relu(y), y)

# Three passes versus one fused pass, N=1024 fp32 (b=4 bytes).
N, itemsize = 1024, 4
q_sep = 6 * N * itemsize       # 3 reads + 3 writes
q_fused = 2 * N * itemsize     # 1 read + 1 write
q_saved = q_sep - q_fused

assert q_sep == 24_576
assert q_fused == 8_192
assert q_saved == 16_384
assert q_sep // q_fused == 3
assert q_saved == 4 * N * itemsize

## Part II — Method 01: separate PyTorch operations

- **Connection.** Instantiates the three-pass baseline counted in §20.9 on the worked-example spine.
- **Rule.** In eager PyTorch, each operator is a separate kernel launch.
- **Baseline.** `t = x * a`, `u = t + b`, and `y = torch.relu(u)` materialize $t$ and $u$ in HBM.
- **Intermediates.** Every intermediate is a full-size tensor with the same shape and dtype as the input.
- **Contract.** This is the correctness oracle for the fused kernels; a compiled backend may fuse the operators, but eager execution is the comparison.
- **Check.** The next cells define the three-kernel reference and compare it with the one-expression reference.

In [ ]:
def separate_affine_relu(x: torch.Tensor, a: float, b: float) -> torch.Tensor:
    """Method 01: three conceptual kernels, two materialized intermediates."""
    t = x * a          # kernel 1: multiply, write t
    u = t + b          # kernel 2: add, write u
    return torch.relu(u)   # kernel 3: activation, write y


def fused_affine_relu_reference(x: torch.Tensor, a: float, b: float) -> torch.Tensor:
    """The same function without the intermediate stores."""
    return torch.relu(a * x + b)


x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_sep = separate_affine_relu(x, 2.0, -1.0)
y_ref = fused_affine_relu_reference(x, 2.0, -1.0)
assert y_sep.tolist() == [1.0, 3.0, 5.0, 7.0]
assert torch.equal(y_sep, y_ref)
assert y_sep.shape == x.shape == (4,)
assert y_sep.dtype == x.dtype == torch.float32
print("Method 01: three eager passes match the fused reference")

In [ ]:
def test_method01_separate() -> None:
    """CPU: the three-pass reference matches the fused expression on random data."""
    torch.manual_seed(1)
    for shape in ((7,), (3, 5), (2, 3, 4)):
        x = torch.randn(*shape)
        for a, b in ((2.0, -1.0), (-0.5, 3.0), (0.0, 1.0)):
            got = separate_affine_relu(x, a, b)
            ref = fused_affine_relu_reference(x, a, b)
            assert got.shape == ref.shape == x.shape
            assert torch.allclose(got, ref, atol=1e-6, rtol=1e-6)
    print("Method 01: shapes and random cases OK")


test_method01_separate()

## Part III — Method 02: fused Triton kernels

- **Connection.** Turns each Part I rule into one masked-load/compose/masked-store kernel, starting from the baseline in Part II.
- **Skeleton.** Masked load → compose in registers → masked store; only the expression between the load and store changes.
- **Kernels.** Affine + ReLU; residual + activation; sum-of-squares + normalization (RMSNorm); a five-operation elementwise chain.
- **Checks.** Each kernel is compared with PyTorch on the active device.
- **CPU path.** On a CPU-only machine the tests print `SKIP ...`; under the Triton interpreter they execute on CPU tensors.

In [ ]:
@triton.jit
def affine_relu_kernel(x_ptr, out_ptr, a, b, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    z = a * x + b
    tl.store(out_ptr + offs, tl.maximum(z, 0.0), mask=mask)


def fused_affine_relu(x: torch.Tensor, a: float, b: float, BLOCK: int = 256) -> torch.Tensor:
    """Method 02: y = ReLU(a*x + b) with one load and one store."""
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    affine_relu_kernel[(triton.cdiv(n, BLOCK),)](x, out, a, b, n, BLOCK=BLOCK)
    return out

In [ ]:
def test_fused_affine_relu() -> None:
    """GPU/interpreter: the fused spine kernel matches PyTorch."""
    require_gpu("fused affine+relu")
    x = torch.linspace(-3.0, 3.0, 1000, device=DEVICE)
    out = fused_affine_relu(x, 2.0, -1.0)
    assert out.shape == x.shape
    check("affine+relu", out, torch.relu(2.0 * x - 1.0), atol=1e-5, rtol=1e-5)


run_if_gpu(test_fused_affine_relu)

In [ ]:
@triton.jit
def residual_relu_kernel(x_ptr, r_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    r = tl.load(r_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, tl.maximum(x + r, 0.0), mask=mask)


def fused_residual_relu(x: torch.Tensor, r: torch.Tensor, BLOCK: int = 256) -> torch.Tensor:
    """Method 02: y = ReLU(x + r); two loads, one store, no intermediate."""
    assert x.shape == r.shape, "residual and input must have the same shape"
    x, r = x.contiguous(), r.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    residual_relu_kernel[(triton.cdiv(n, BLOCK),)](x, r, out, n, BLOCK=BLOCK)
    return out

In [ ]:
def test_fused_residual() -> None:
    """GPU/interpreter: the fused residual+ReLU matches PyTorch."""
    require_gpu("fused residual+relu")
    torch.manual_seed(2)
    x = torch.randn(1000, device=DEVICE)
    r = torch.randn(1000, device=DEVICE)
    out = fused_residual_relu(x, r)
    assert out.shape == x.shape
    check("residual+relu", out, torch.relu(x + r), atol=1e-5, rtol=1e-5)


run_if_gpu(test_fused_residual)

In [ ]:
@triton.jit
def rms_norm_kernel(x_ptr, out_ptr, n_elements, eps, BLOCK: tl.constexpr):
    offs = tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    mean_sq = tl.sum(x * x, axis=0) / n_elements
    inv_rms = tl.rsqrt(mean_sq + eps)
    tl.store(out_ptr + offs, x * inv_rms, mask=mask)


def fused_rms_norm(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Method 02: y = x / sqrt(mean(x^2) + eps), one row per program."""
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    rms_norm_kernel[(1,)](x, out, n, eps, BLOCK=triton.next_power_of_2(n))
    return out

In [ ]:
def test_fused_rms_norm() -> None:
    """GPU/interpreter: sum-of-squares + normalization matches PyTorch."""
    require_gpu("fused rms norm")
    torch.manual_seed(3)
    x = torch.randn(257, device=DEVICE)
    out = fused_rms_norm(x)
    ref = x * torch.rsqrt((x * x).sum() / x.numel() + 1e-6)
    assert out.shape == x.shape
    check("rms norm", out, ref, atol=1e-5, rtol=1e-5)


run_if_gpu(test_fused_rms_norm)

In [ ]:
@triton.jit
def chain_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = x + 1.0
    y = y * 2.0
    y = tl.abs(y)
    y = tl.sqrt(y + 1.0)
    y = 1.0 / (1.0 + y)
    tl.store(out_ptr + offs, y, mask=mask)


def fused_chain(x: torch.Tensor, BLOCK: int = 256) -> torch.Tensor:
    """Method 02: five pointwise operations in one load/store pair."""
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    chain_kernel[(triton.cdiv(n, BLOCK),)](x, out, n, BLOCK=BLOCK)
    return out

In [ ]:
def test_fused_chain() -> None:
    """GPU/interpreter: the five-op chain matches PyTorch."""
    require_gpu("fused chain")
    torch.manual_seed(4)
    x = torch.randn(1000, device=DEVICE)
    out = fused_chain(x)
    ref = 1.0 / (1.0 + torch.sqrt(torch.abs((x + 1.0) * 2.0) + 1.0))
    assert out.shape == x.shape
    check("five-op chain", out, ref, atol=1e-6, rtol=1e-6)


run_if_gpu(test_fused_chain)

## Part IV — Method 03: a GEMM epilogue with `tl.dot`

- **Connection.** Extends the fused-tile idea from §20.7–§20.8 to the matrix-product accumulator.
- **Composition.** $C=\operatorname{act}(AB+\mathbf 1 b^\top)$.
- **Epilogue.** `tl.dot` accumulates a $B_M\times B_N$ tile in fp32 registers; bias and activation run after the final accumulation and before the store, so $AB$ never exists in HBM.
- **Tile constraint.** `tl.dot` requires dimensions of at least 16; the 2×2 LaTeX example in §20.7 is checked on the CPU with PyTorch, and the kernel test uses a 16×16 integer-valued matrix where fp32/TF32 is exact.
- **Register note.** The $B_M\times B_N$ accumulator is live across the whole $K$ loop; a few epilogue temporaries are cheap, but a second GEMM-sized intermediate doubles the live footprint and starts spilling.

In [ ]:
@triton.jit
def gemm_epilogue_kernel(A, B, bias, C, M, N, K, ACT: tl.constexpr,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
                         BLOCK_K: tl.constexpr):
    pid = tl.program_id(0)
    num_n = tl.cdiv(N, BLOCK_N)
    pm = pid // num_n
    pn = pid % num_n

    rows = pm * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pn * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, tl.cdiv(K, BLOCK_K)):
        rk = k * BLOCK_K + tl.arange(0, BLOCK_K)
        a = tl.load(A + rows[:, None] * K + rk[None, :],
                    mask=(rows[:, None] < M) & (rk[None, :] < K), other=0.0)
        b = tl.load(B + rk[:, None] * N + cols[None, :],
                    mask=(rk[:, None] < K) & (cols[None, :] < N), other=0.0)
        acc = tl.dot(a, b, acc=acc)

    bv = tl.load(bias + cols, mask=cols < N, other=0.0).to(tl.float32)
    acc = acc + bv[None, :]
    if ACT == 1:
        acc = tl.maximum(acc, 0.0)

    tl.store(C + rows[:, None] * N + cols[None, :], acc,
             mask=(rows[:, None] < M) & (cols[None, :] < N))


def gemm_epilogue(A: torch.Tensor, B: torch.Tensor, bias: torch.Tensor,
                  act: int = 0) -> torch.Tensor:
    """Method 03: C = act(A @ B + bias), fused in the epilogue."""
    A, B, bias = A.contiguous(), B.contiguous(), bias.contiguous()
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, "inner dimensions must match"
    assert bias.shape == (N,), "bias must be a length-N row"
    C = torch.empty((M, N), device=A.device, dtype=torch.float32)
    BM = BN = BK = 16
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)
    gemm_epilogue_kernel[grid](A, B, bias, C, M, N, K, ACT=act,
                               BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
    return C

In [ ]:
# The tiny 2x2 values from the LaTeX example, checked on the CPU.
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.eye(2)
bias = torch.tensor([10.0, 20.0])
C = A @ B + bias
assert C.tolist() == [[11.0, 22.0], [13.0, 24.0]]

A2 = torch.tensor([[1.0, -1.0], [-2.0, 3.0]])
C2 = torch.relu(A2 @ torch.eye(2))
assert C2.tolist() == [[1.0, 0.0], [0.0, 3.0]]
assert torch.equal(torch.relu(C2), C2)

In [ ]:
def test_gemm_epilogue() -> None:
    """GPU/interpreter: GEMM+bias and GEMM+bias+ReLU match PyTorch."""
    require_gpu("gemm epilogue")
    torch.manual_seed(5)
    # Integer-valued matrices keep the fp32/TF32 dot product exact.
    A = torch.randint(-2, 3, (16, 16), device=DEVICE).float()
    B = torch.randint(-2, 3, (16, 16), device=DEVICE).float()
    bias = torch.randint(-2, 3, (16,), device=DEVICE).float()

    ref = A @ B + bias
    out = gemm_epilogue(A, B, bias, act=0)
    assert out.shape == (16, 16)
    torch.testing.assert_close(out, ref, atol=1e-4, rtol=1e-4)

    out_relu = gemm_epilogue(A, B, bias, act=1)
    torch.testing.assert_close(out_relu, torch.relu(ref), atol=1e-4, rtol=1e-4)
    print("gemm epilogue: bias and bias+ReLU match")


run_if_gpu(test_gemm_epilogue)

### Register pressure in the epilogue

- **Connection.** Applies the $r\approx LE/T$ model from §20.10 to the GEMM accumulator.
- **Accumulator cost.** $r_{\text{acc}}=B_MB_N/T$ registers per thread before any epilogue temporary.
- **Example.** For $B_M=B_N=128$ and $T=256$: $r_{\text{acc}}=64$ registers per thread.
- **Second matrix.** A second saved matrix of the same size doubles that; a third exceeds the 255-register budget even before loop indices and pointers.
- **Practical rule.** Fuse operations that consume the accumulator in place; do not fuse operations that require a second GEMM-sized intermediate.
- **Fallback.** If the epilogue needs both the pre-activation and the post-activation, store one and recompute the other, or split the kernel.
- **Check.** The next cell verifies the register arithmetic used here and in §20.10; it does not launch a kernel.

In [ ]:
# Register model used in the text: r = ceil(L * E / T), spill if r > 255.
def regs_per_thread(live_tiles: int, tile_elems: int, threads: int) -> int:
    return -(-live_tiles * tile_elems // threads)


assert regs_per_thread(1, 1024, 128) == 8
assert regs_per_thread(8, 1024, 128) == 64
assert regs_per_thread(32, 1024, 128) == 256
assert regs_per_thread(32, 1024, 128) > 255
assert regs_per_thread(32, 1024, 256) == 128
assert regs_per_thread(128 * 128, 1, 256) == 64

N, itemsize = 1024, 4
q_base = 2 * N * itemsize
q_spill = q_base + 2 * N * itemsize
assert q_base == 8_192
assert q_spill == 16_384
assert q_spill == 2 * q_base

## When not to fuse

- **Connection.** Qualifies the byte-model decision from §20.9 with the register limit from §20.10.
- **Rule.** Fusion is a traffic optimization, not a universal speedup; when extra live state or a changed schedule costs more than the saved bytes, split.
- **Spill model.** With $s$ spilled tiles out of $L$ live tiles, $Q_{\text{fuse}}=2Nb$ when $L\le L_{\max}$, otherwise $Q_{\text{fuse}}=2Nb+2sNb$; $Q_{\text{sep}}=2kNb$.
- **Crossover.** Fusion wins while $2+2s\lt 2k$, that is, while $s\lt k-1$. For a two-pass chain one spill is a tie; for $k=3$, one spill still wins and two spills lose.

| Situation | Why fusion can lose | What to do |
|---|---|---|
| Register spills | A spilled tile adds $2Nb$ local-memory traffic per tile; occupancy drops | Reduce `BLOCK`, split the chain, recompute a cheap value |
| Low occupancy | A wide fused kernel may fit fewer blocks per SM and hide less latency | Increase `num_warps` or use a smaller tile |
| Recomputation | Recomputing an expensive intermediate in a backward pass can cost more than storing it | Save the intermediate, or use a two-kernel forward |
| Code size | A long chain or many `if` branches bloats the instruction stream | Split at a natural boundary |
| Numerical differences | Reassociating a sum or fusing an FMA changes the last bits | Match the reference order when bitwise reproducibility matters |
| Aliasing / in-place | Writing over a buffer that a later expression still reads is a race or a wrong result | Use a separate output, or prove the lifetime |

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Saving an intermediate that is still needed | The next expression reads a tile that was overwritten or freed | Compute the output last; keep the pre-activation only if the backward needs it |
| Broadcasting the bias wrong | `bias[:, None]` makes a per-row bias instead of a per-column bias | Use `bias[None, :]` for a row vector; assert the shape |
| Fusing a reduction incorrectly | `tl.sum` over the wrong axis, or `other=0.0` wins a masked maximum | Reduce the intended axis; use `-inf` for max, `+inf` for min, `0` for sum |
| Register pressure | `r > 255` spills; a "one-pass" kernel becomes two reads plus two writes | Reduce `BLOCK`, split the chain, or recompute |
| Reading and writing the same buffer | In-place update changes the input of a later lane or a later pass | Use distinct input and output pointers; check aliasing |
| Numerical reassociation | Fused FMA or a different summation order changes the last bits | Match the reference order, or document the tolerance |
| Counting a scalar as `Nb` | The predicted speedup is too large | Count full-size streams only |
| Fusing before measuring | The kernel is slower because occupancy fell | Model the bytes, then benchmark both versions |

## Self-check

1. For $x=(1,2,3,4)$, $a=2$, $b=-1$, write $y=a x+b$ and
   $\operatorname{ReLU}(y)$. Which entries change under the ReLU?
2. Derive $Q_{\text{sep}}$ and $Q_{\text{fused}}$ for a four-pass unary chain at
   $N=2048$ fp32. How many bytes are saved?
3. Why does a bias add cost $5Nb$ unfused but only $3Nb$ fused, while a scalar
   affine costs $4Nb$ unfused and $2Nb$ fused?
4. In the sum-of-squares normalization, why is the fused kernel's traffic
   $2Nb$ rather than $3Nb$? What does it hold in registers?
5. A GEMM epilogue adds a length-$N$ bias and a ReLU. Which matrix never reaches
   HBM, and why does that save $2MNb$?
6. A fused kernel with $E=2048$, $T=128$, and $L=20$ uses how many registers per
   thread? Does it spill?
7. Give one case where saving an intermediate is better than recomputing it.
8. Why can two fused summations of the same numbers differ in the last bits?

## Summary

- **Recurring idea.** Fusion moves intermediates from HBM to registers; the byte model is the first tool.
- **Traffic rule.** $Q_{\text{sep}}=2kNb$ and $Q_{\text{fused}}=(I+O)Nb$; for a unary chain the ratio is $k$.
- **Ceiling, not guarantee.** Cache residency, launch overhead, occupancy, and spills move the measured result.
- **Method 01.** One eager PyTorch kernel per operation; every intermediate is materialized.
- **Method 02.** Pointwise maps, residual adds, and a reduction compose in one kernel with one load and one store.
- **Method 03.** Bias and activation run in the GEMM epilogue on the fp32 accumulator before the only store.
- **Decision.** Fuse while the live state fits in registers and the saved traffic exceeds any spill cost; otherwise split.

## Key takeaways

- One load and one store per element is the target.
- A reduction can be fused with its normalization because the tile is already in registers.
- The GEMM epilogue is the highest-value fusion in deep learning.
- Register pressure is the limit that the byte model does not show.

**Where this leads.**
- **Chapter 21.** Atomics, synchronization, and race conditions for updates shared across programs.
- **Chapter 22.** Full `tl.dot` GEMMs, tensor cores, and epilogue optimization.
- **Chapter 23.** Fused transformer blocks that chain attention, normalization, and MLP epilogues.
- **Chapter 24.** Profiling and roofline analysis to confirm whether the fused kernel is bandwidth- or occupancy-limited.

## Exercises

1. **Four-pass traffic.** For $N=2048$ fp32, derive the bytes of a four-pass
   unary chain and of one fused pass. Verify the ratio with assertions.
2. **Bias axis.** Write a 2-D bias+ReLU kernel with `bias[None, :]` and then with
   `bias[:, None]`. Construct a shape where the second silently broadcasts, and
   explain the wrong result.
3. **Residual aliasing.** Modify `fused_residual_relu` to write in place over
   `x`. Find an input where the result differs from the out-of-place version.
4. **Chain split.** Split the five-op chain into two kernels at the square root.
   Count the bytes of the split and compare with the fused version.
5. **Reduction axis.** Write a 2-D row normalization that divides each row by its
   own sum. Test a non-square matrix and a masked last row.
6. **Softmax epilogue.** Extend the fused reduction kernel to a stable softmax
   (subtract the row maximum, exponentiate, divide) with one load and one store.
7. **GEMM epilogue.** Add a GELU activation to `gemm_epilogue_kernel`. Use the
   tanh approximation and explain why the interpreter may need a fallback.
8. **Register budget.** For $B_M=B_N=64$, $T=128$, compute the accumulator
   registers per thread. How many additional $64\times64$ fp32 tiles fit under
   255 registers?
9. **Spill crossover.** Using $Q_{\text{fuse}}(L)$ from §20.10, find the smallest
   $s$ that makes a three-pass chain prefer the separate kernels.
10. **Measure.** Benchmark the fused and separate affine+ReLU on a GPU. Compare
    the measured ratio with the predicted $3\times$ byte ratio and explain the
    gap.

In [ ]:
CPU_TESTS = [
    test_method01_separate,
]

GPU_TESTS = [
    test_fused_affine_relu,
    test_fused_residual,
    test_fused_rms_norm,
    test_fused_chain,
    test_gemm_epilogue,
]

print("CPU checks")
for test in CPU_TESTS:
    test()

print()
print("GPU checks")
for test in GPU_TESTS:
    run_if_gpu(test)

print()
print("done: CPU checks passed; GPU checks ran or printed SKIP")